# Mathematics for Machine Learning: Quiz Answers

**Data:** the *tips* data set has 244 restaurant bills with the columns `total_bill` (\$), `tip` (\$), `sex`, `smoker`, `day`, `time` and `size` (number of people).
It was downloaded from https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv and is provided as `tips.csv`.

We want to predict the tip from the information on the bill with least squares models.

> The answers does't have to be exactly the same as the ones provided here, but they should be equivalent.

In [1]:
import numpy as np

data = np.genfromtxt(
    "tips.csv", delimiter=",", names=True, dtype=None, encoding="utf-8"
)
print(data.dtype.names)
print(data[:3])

('total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size')
[(16.99, 1.01, '"Female"', '"No"', '"Sun"', '"Dinner"', 2)
 (10.34, 1.66, '"Male"', '"No"', '"Sun"', '"Dinner"', 3)
 (21.01, 3.5 , '"Male"', '"No"', '"Sun"', '"Dinner"', 3)]


### (a) Vectors and 0/1 features
Build the vectors `bill`, `tip` and `size`, and the 0/1 vectors `smoker` (1 if the party contains a smoker) and `dinner` (1 if `time` is Dinner).

*Hint:* text entries keep their quotation marks. A 0/1 vector is built by a comparison, e.g.
`smoker = (data["smoker"] == '"Yes"').astype(float)`

In [2]:
# (a) your code here

bill, tip, size = data["total_bill"], data["tip"], data["size"].astype(float)
smoker = (data["smoker"] == '"Yes"').astype(float)
dinner = (data["time"] == '"Dinner"').astype(float)

### (b) 80% / 20% split
Split the data into 80% training and 20% test data.

In [3]:
# (b) your code here

N = len(data)
np.random.seed(42)
idx = np.random.permutation(N)
n_train = int(0.8 * N)
train, test = idx[:n_train], idx[n_train:]

### (c) Three least squares models
Fit the following models by least squares on the training set. For each model print the parameter vector $\hat\theta$ and the RMS prediction error on the training set and on the test set:

1. $[\mathbf{1},\ \text{bill}]$
2. $[\mathbf{1},\ \text{bill},\ \text{size}]$
3. $[\mathbf{1},\ \text{bill},\ \text{size},\ \text{smoker},\ \text{dinner}]$

In [4]:
# (c) your code here


def rms(v):
    return np.sqrt(np.mean(v**2))


one = np.ones(N)
models = {
    "[1, bill]": np.column_stack([one, bill]),
    "[1, bill, size]": np.column_stack([one, bill, size]),
    "[1, bill, size, smoker, dinner]": np.column_stack(
        [one, bill, size, smoker, dinner]
    ),
}

thetas, test_errors = {}, {}
for name, A in models.items():
    th = np.linalg.lstsq(A[train], tip[train], rcond=None)[0]
    thetas[name], test_errors[name] = th, rms(tip[test] - A[test] @ th)
    print(
        name,
        th.round(4),
        "train",
        rms(tip[train] - A[train] @ th).round(4),
        "test",
        test_errors[name].round(4),
    )

[1, bill] [0.9315 0.1054] train 0.9545 test 1.239
[1, bill, size] [0.6704 0.0915 0.2084] train 0.9409 test 1.2375
[1, bill, size, smoker, dinner] [0.6304 0.0905 0.2137 0.0243 0.051 ] train 0.9405 test 1.242


### (d) Prediction
A party of 4 people, with no smokers, has dinner and the total bill is $40. Write the feature vector of this party for each of the three models. Select the model with the smallest test RMS error in your code (e.g. with `np.argmin`), and print its name and its predicted tip.

In [5]:
# (d) your code here

x_new = {
    "[1, bill]": np.array([1, 40]),
    "[1, bill, size]": np.array([1, 40, 4]),
    "[1, bill, size, smoker, dinner]": np.array([1, 40, 4, 0, 1]),
}
names = list(models)
best = names[np.argmin([test_errors[n] for n in names])]
print("best model:", best, " predicted tip:", (x_new[best] @ thetas[best]).round(2))

best model: [1, bill, size]  predicted tip: 5.16
